# Writing Effective Prompts for Agents: Practice Exercise (Solution)

In this exercise, you will write the system prompt for a ReAct (Reasoning and Acting) agent. The agent code structure and tools are provided - your task is to craft an effective prompt that guides the agent's behavior.

**What you'll implement:**
- A system prompt that instructs the agent to follow the Thought -> Action -> PAUSE -> Observation loop
- Documentation of available tools within the prompt
- An example interaction to guide the agent's behavior

**Estimated time:** 10-15 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [1]:
# Setup - run this cell first

from openai import OpenAI
import re
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

client = OpenAI()

print("Setup complete!")

Setup complete!


## Context

You are building a **book recommendation agent** that helps users find books based on genre and get reading time estimates. The agent uses the ReAct pattern, which means it must:

1. **Think** about what it needs to do
2. Take an **Action** using one of its tools
3. **PAUSE** to wait for the result
4. **Observe** the result and decide next steps
5. Repeat until it can provide a final **Answer**

**Available Tools:**
- `get_book_recommendation`: Takes a genre and returns a book recommendation with page count
- `estimate_reading_time`: Takes a page count and returns estimated reading hours

**Your task:** Write a system prompt that teaches the agent how to use these tools in the ReAct loop format.

**Expected agent output format:**
```
Thought: I need to find a book in the mystery genre first.
Action: get_book_recommendation: mystery
PAUSE
```

After receiving an observation, the agent continues reasoning until it can provide a final answer.

## Provided Code: Agent Class and Tools

The following code is provided for you. Review it to understand how the agent and tools work.

In [2]:
# Agent class - provided for you

class BookAgent:
    """
    A ReAct agent that uses OpenAI to reason and act.
    The agent's behavior is determined by the system prompt.
    """

    def __init__(self, system_prompt: str):
        """
        Initialize the agent with a system prompt.

        Args:
            system_prompt: Instructions that define the agent's behavior
        """
        self.system_prompt = system_prompt
        self.messages = []
        if self.system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def __call__(self, message: str) -> str:
        """
        Send a message to the agent and get a response.

        Args:
            message: The user's input message

        Returns:
            The agent's response
        """
        self.messages.append({"role": "user", "content": message})
        result = self._get_completion()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def _get_completion(self) -> str:
        """Get a completion from the OpenAI API."""
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=self.messages
        )
        return completion.choices[0].message.content

In [3]:
# Tools - provided for you

# Book database with page counts
book_database = {
    "mystery": {"title": "The Silent Patient", "author": "Alex Michaelides", "pages": 336},
    "science fiction": {"title": "Project Hail Mary", "author": "Andy Weir", "pages": 496},
    "fantasy": {"title": "The Name of the Wind", "author": "Patrick Rothfuss", "pages": 662},
    "romance": {"title": "Beach Read", "author": "Emily Henry", "pages": 361},
    "thriller": {"title": "Gone Girl", "author": "Gillian Flynn", "pages": 415}
}

def get_book_recommendation(genre: str) -> str:
    """
    Get a book recommendation for a given genre.
    
    Args:
        genre: The book genre (mystery, science fiction, fantasy, romance, thriller)
    
    Returns:
        A string with the book recommendation and page count
    """
    genre = genre.lower().strip()
    if genre in book_database:
        book = book_database[genre]
        return f"'{book['title']}' by {book['author']} ({book['pages']} pages)"
    else:
        return f"Sorry, I don't have recommendations for the '{genre}' genre. Available genres: mystery, science fiction, fantasy, romance, thriller."


def estimate_reading_time(pages: str) -> str:
    """
    Estimate reading time based on page count.
    Assumes average reading speed of 30 pages per hour.
    
    Args:
        pages: Number of pages as a string
    
    Returns:
        A string with the estimated reading time
    """
    try:
        page_count = int(pages)
        hours = page_count / 30
        return f"Estimated reading time for {page_count} pages: {hours:.1f} hours"
    except ValueError:
        return "Error: Please provide a valid number of pages."


# Map action names to functions
available_actions = {
    "get_book_recommendation": get_book_recommendation,
    "estimate_reading_time": estimate_reading_time
}

print("Tools loaded successfully!")
print(f"Available actions: {list(available_actions.keys())}")

Tools loaded successfully!
Available actions: ['get_book_recommendation', 'estimate_reading_time']


## Solution: The Agent Prompt

Below is a working solution for the ReAct agent prompt.

In [4]:
def create_react_prompt() -> str:
    """
    Create a system prompt for the book recommendation ReAct agent.
    
    The prompt should instruct the agent to:
    - Use the Thought -> Action -> PAUSE -> Observation loop
    - Document both available actions (get_book_recommendation, estimate_reading_time)
    - Include usage examples for each action
    - Provide a complete example session showing the loop in action
    - End with Answer: when the task is complete
    
    Action format must be: Action: action_name: argument
    Example: Action: get_book_recommendation: mystery
    
    Returns:
        str: The complete system prompt for the ReAct agent
    """
    prompt = """
You run in a loop of Thought, Action, PAUSE, and Observation.
At the end of the loop you output an Answer.

Use Thought to describe your reasoning about what you need to do next.
Use Action to run one of the available actions - then return PAUSE.
Observation will be the result of running those actions (provided by the system).

Your available actions are:

get_book_recommendation:
e.g. get_book_recommendation: fantasy
Returns a book recommendation for the given genre, including the title, author, and page count.
Available genres: mystery, science fiction, fantasy, romance, thriller.

estimate_reading_time:
e.g. estimate_reading_time: 350
Returns the estimated reading time in hours for a given number of pages.

Example session:

Question: What science fiction book do you recommend and how long will it take to read?
Thought: I need to first get a science fiction book recommendation, then estimate the reading time based on its page count.
Action: get_book_recommendation: science fiction
PAUSE

Observation: 'Project Hail Mary' by Andy Weir (496 pages)

Thought: Now I have the book recommendation with 496 pages. I should estimate the reading time for 496 pages.
Action: estimate_reading_time: 496
PAUSE

Observation: Estimated reading time for 496 pages: 16.5 hours

Thought: I now have both the book recommendation and the reading time estimate. I can provide the final answer.
Answer: I recommend 'Project Hail Mary' by Andy Weir. It's 496 pages and will take approximately 16.5 hours to read.
"""
    
    return prompt.strip()

## Provided Code: Query Function

This function runs the ReAct loop, executing actions and feeding observations back to the agent.

In [5]:
# Query function - provided for you

# Regex to parse action from agent output
action_pattern = re.compile(r'^Action: (\w+): (.*)$', re.MULTILINE)


def run_agent(question: str, max_iterations: int = 5) -> None:
    """
    Run the ReAct agent loop for a given question.
    
    Args:
        question: The user's question
        max_iterations: Maximum number of action-observation cycles
    """
    # Create agent with your prompt
    prompt = create_react_prompt()
    agent = BookAgent(prompt)
    
    print(f"Question: {question}")
    print("=" * 60)
    
    iteration = 0
    next_input = question
    
    while iteration < max_iterations:
        iteration += 1
        response = agent(next_input)
        print(response)
        
        # Look for actions in the response
        actions = action_pattern.findall(response)
        
        if actions:
            # Execute the first action found
            action_name, action_input = actions[0]
            
            if action_name not in available_actions:
                print(f"\n[ERROR] Unknown action: {action_name}")
                break
            
            # Execute the action
            print(f"\n[Executing: {action_name}({action_input})]")
            observation = available_actions[action_name](action_input)
            print(f"Observation: {observation}\n")
            
            # Feed the observation back to the agent
            next_input = f"Observation: {observation}"
        else:
            # No actions found - agent is done
            break
    
    print("=" * 60)

## Test Your Prompt

Run these test cases to verify your prompt works correctly.

In [6]:
# Test 1: Simple book recommendation
run_agent("Can you recommend a mystery book?")

Question: Can you recommend a mystery book?
Thought: I need to get a book recommendation for the mystery genre. 
Action: get_book_recommendation: mystery
PAUSE

[Executing: get_book_recommendation(mystery)]
Observation: 'The Silent Patient' by Alex Michaelides (336 pages)

Thought: I have the book recommendation 'The Silent Patient' by Alex Michaelides, which has 336 pages. Now, I should estimate the reading time for 336 pages. 
Action: estimate_reading_time: 336
PAUSE

[Executing: estimate_reading_time(336)]
Observation: Estimated reading time for 336 pages: 11.2 hours

Thought: I now have both the book recommendation and the estimated reading time. I can provide the final answer. 
Answer: I recommend 'The Silent Patient' by Alex Michaelides. It's 336 pages and will take approximately 11.2 hours to read.


In [7]:
# Test 2: Book with reading time estimate (requires both tools)
run_agent("I want to read a fantasy book. How long will it take me to finish?")

Question: I want to read a fantasy book. How long will it take me to finish?
Thought: I need to first get a fantasy book recommendation, then estimate the reading time based on its page count. 
Action: get_book_recommendation: fantasy 
PAUSE

[Executing: get_book_recommendation(fantasy )]
Observation: 'The Name of the Wind' by Patrick Rothfuss (662 pages)

Thought: Now I have the book recommendation with 662 pages. I should estimate the reading time for 662 pages. 
Action: estimate_reading_time: 662 
PAUSE

[Executing: estimate_reading_time(662 )]
Observation: Estimated reading time for 662 pages: 22.1 hours

Thought: I now have both the book recommendation and the reading time estimate. I can provide the final answer. 
Answer: I recommend 'The Name of the Wind' by Patrick Rothfuss. It's 662 pages and will take approximately 22.1 hours to read.
